In [ ]:
!pip install py3Dmol rdkit

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload targets.json and docked_poses.sdf

In [ ]:
import json
import py3Dmol
from rdkit import Chem

with open('targets.json') as f:
    targets = json.load(f)

suppl = Chem.SDMolSupplier('docked_poses.sdf', removeHs=False)
mols = {m.GetProp('_Name'): m for m in suppl if m}

colors = {
    'Donor': 'blue',
    'Acceptor': 'red',
    'Hydrophobe': 'orange',
    'Aromatic': 'green'
}

for name, target in targets.items():
    mol = mols.get(name)
    if mol is None:
        print(f'{name}: not found in SDF')
        continue

    sdf_str = Chem.MolToMolBlock(mol)

    view = py3Dmol.view(width=700, height=450)
    view.addModel(sdf_str, 'sdf')
    view.setStyle({'stick': {}, 'sphere': {'radius': 0.25}})

    for site in target['interaction_sites']:
        view.addSphere({
            'center': {'x': site['x'], 'y': site['y'], 'z': site['z']},
            'radius': 0.5,
            'color': colors.get(site['family'], 'white'),
            'opacity': 0.6
        })

    for ev in target.get('excluded_volumes', []):
        view.addSphere({
            'center': {'x': ev['x'], 'y': ev['y'], 'z': ev['z']},
            'radius': ev.get('radius', 1.2),
            'color': 'gray',
            'opacity': 0.2
        })

    print(f'\n── {name} ──')
    view.zoomTo()
    view.show()